# DX 704 Week 13 Project

In this project, you will use a language model API to turn unstructured text into data, then build a simple AI agent that analyzes the resulting dataset and writes a report.


You will work with fictional city service requests. Each request contains a resident's description of a problem, along with information recorded by the city's intake system. The intake classifications are imperfect: an apparently routine request might describe a significant safety hazard, and different residents may describe similar problems in very different ways.

You will complete the following task:
Give a language model access to data-analysis tools so it can investigate the enriched dataset and prepare a quantitative report.
You may use AI coding assistants (including ChatGPT, Codex, or Claude Code) to help write and debug your code, but you are responsible for verifying the outputs and conclusions.

The full project description and a template notebook are available on GitHub: [Project 13 Materials](https://github.com/bu-cds-dx704/dx704-project-13).


### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

### Documentation

The following OpenAI API documentation will be helpful.

- [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Responses API](https://developers.openai.com/api/reference/resources/responses/methods/create)
- [Function Calling](https://developers.openai.com/api/docs/guides/function-calling)

### Modules

In [1]:
%pip install -q -U duckdb matplotlib openai pydantic pandas tqdm ipywidgets


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import datetime
import decimal
import hashlib
import json
import math
import os
import pathlib
import re
import time
import typing
import uuid

import duckdb
import matplotlib.pyplot as plt
import openai
import pandas as pd
import pydantic
import tqdm.notebook
import IPython
import concurrent.futures

### Raw Data

You are given the following files:

| File | Contents |
|---|---|
| `requests.csv` | Approximately 2,000 service requests with `request_id`, `created_at`, `closed_at`, `district_id`, `description`, `intake_category`, `intake_priority`, `status`, and other operational fields |
| `districts.csv` | District names and populations |
| `events.csv` | Weather and operational events that might help explain request patterns |

In [3]:
!wget -nc https://github.com/bu-cds-dx704/dx704-project-13/raw/main/requests.csv
!head requests.csv

File ‘requests.csv’ already there; not retrieving.

request_id,incident_id,created_at,closed_at,district_id,report_channel,status,closure_reason,intake_category,intake_priority,description
R000048,I021307,2025-01-01 18:52:00,2025-01-02 12:19:07,D07,Phone,Closed,Resolved,Road Maintenance,Routine,There is standing water by the bus stop after every shower. This is opposite the pharmacy.
R000092,I016489,2025-01-02 06:15:00,2025-01-03 08:17:24,D02,Phone,Closed,Resolved,Road Maintenance,Routine,"Just sending a note about a problem. Beside the school entrance, the old road patch is coming apart again; it needs another look."
R000103,I015189,2025-01-02 10:42:00,2025-01-07 09:57:08,D04,Phone,Closed,Resolved,Drainage,High,The blocked inlet has left the school approach under deep water by the bicycle racks.
R000176,I006845,2025-01-03 12:29:00,2025-01-04 08:28:50,D04,Web,Closed,Resolved,Road Maintenance,High,I do not know which office handles it. The bike lane has broken pavement that makes for a 

In [4]:
!wget -nc https://github.com/bu-cds-dx704/dx704-project-13/raw/main/districts.csv
!head districts.csv

File ‘districts.csv’ already there; not retrieving.

district_id,district_name,population,area_sq_km,road_miles,median_income_usd,housing_units
D01,North Harbor,74000,13.4,106.2,69000,29200
D02,Eastbank,90000,16.7,124.8,58000,34900
D03,Cedar Hills,61000,20.5,142.6,98000,23600
D04,Old Port,84000,11.8,92.4,74000,32400
D05,Riverside,79000,18.2,128.9,62000,30300
D06,Southpoint,95000,15.6,135.7,55000,38100
D07,West Commons,68000,19.1,118.5,82000,26500
D08,Lakeview,85000,21.9,149.3,76000,33000


In [5]:
!wget -nc https://github.com/bu-cds-dx704/dx704-project-13/raw/main/events.csv
!head events.csv
!wget -nc https://github.com/bu-cds-dx704/dx704-project-13/raw/main/validation_labels.csv

File ‘events.csv’ already there; not retrieving.



event_id,start_date,end_date,district_id,event_type,intensity,description
E001,2025-01-03,2025-01-04,D04,Construction,3,Major street or utility construction activity
E002,2025-01-05,2025-01-08,D08,Community Event,4,High attendance at a neighborhood event
E003,2025-01-08,2025-01-09,D01,Construction,4,Major street or utility construction activity
E004,2025-01-08,2025-01-08,D02,Construction,3,Major street or utility construction activity
E005,2025-01-11,2025-01-11,D06,Rainstorm,3,Heavy precipitation recorded in the district
E006,2025-01-13,2025-01-13,D01,Rainstorm,2,Heavy precipitation recorded in the district
E007,2025-01-17,2025-01-20,D08,Cold Snap,4,Freezing conditions affect outdoor infrastructure
E008,2025-01-19,2025-01-19,D08,Construction,2,Major street or utility construction activity
E009,2025-01-20,2025-01-21,D06,Heatwave,3,An extended period of high temperatures
File ‘validation_labels.csv’ already there; not retrieving.



# Part 0: OpenAI Structured Outputs Example

Review the code below. It demonstrates how an OpenAI model can convert a  free-text description into fields with specified types and allowed values.



### API Access

You should use a secure method to provide your API key (for example, environment variables or Colab secrets). Do not put API keys directly in your submitted notebook.

In [6]:
# OpenAI API key
#
# if not using Colab, comment out these lines.

from getpass import getpass
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")

In [7]:
client = openai.OpenAI()  # Reads OPENAI_API_KEY from the environment
MODEL = "gpt-6-luna"  # You may choose another supported OpenAI model.

### Structured Response Queries

In [8]:
class IncidentExtraction(pydantic.BaseModel):
    incident_type: typing.Literal[
        "road", "electrical", "water", "drainage",
        "sanitation", "tree", "traffic_signal", "other", "unknown"
    ]
    urgency: typing.Literal["routine", "urgent", "unknown"]
    safety_hazard: typing.Literal["yes", "no", "unknown"]
    evidence: str

response = client.responses.parse(
    model=MODEL,
    input=[
        {
            "role": "system",
            "content": (
                "Classify the described incident using only the evidence in the text. "
                "Use 'unknown' if there is not enough information. "
                "The evidence field should briefly identify text supporting the labels."
            ),
        },
        {
            "role": "user",
            "content": "A truck hit the traffic light. It is hanging over the road.",
        },
    ],
    text_format=IncidentExtraction,
)

print(response.output_parsed)

incident_type='traffic_signal' urgency='urgent' safety_hazard='yes' evidence='A truck hit the traffic light, which is hanging over the road.'


The model produces a structured object whose fields conform to the supplied schema. That does **not** guarantee the classifications are correct; you will evaluate them in Part 2.

In [9]:
ENRICH_CACHE = pathlib.Path("enrichment_cache.jsonl")

ENRICH_SYSTEM_PROMPT = (
    "You classify city service requests written by residents. "
    "Use only the evidence in the description. Do not rely on any intake category. "
    "incident_type: road (potholes, pavement, sidewalks, paving stones, sinkholes), "
    "electrical (streetlights, exposed or burning wiring, poles, control panels), "
    "water (water mains, leaks, spraying water, water service), "
    "drainage (blocked drains or inlets, flooding, standing water, backups), "
    "sanitation (trash, bins, litter, yard waste, dumping), "
    "tree (branches, limbs, fallen or overgrown trees), "
    "traffic_signal (traffic lights and signals), "
    "other (a clear problem that fits none of these), "
    "unknown (the text is too vague to tell). "
    "urgency: urgent if the problem needs prompt action because it is getting worse, "
    "blocking access, or putting people or property at near-term risk; routine otherwise; "
    "unknown if it cannot be judged. "
    "safety_hazard: yes if the described condition could plausibly injure people or damage "
    "property (for example exposed wiring, burning smells, cars losing control, falling objects, "
    "water entering buildings, sinkholes, dark or failed signals); no if it is a nuisance or "
    "cosmetic issue; unknown if the text does not say enough. "
    "Urgency and safety_hazard are judged separately. "
    "evidence: quote or closely paraphrase the few words that support the labels."
)

def classify_request(request_id, description):
    last_error = None
    for attempt in range(6):
        try:
            response = client.responses.parse(
                model=MODEL,
                input=[
                    {"role": "system", "content": ENRICH_SYSTEM_PROMPT},
                    {"role": "user", "content": description},
                ],
                text_format=IncidentExtraction,
            )
            return {"request_id": request_id, **response.output_parsed.model_dump()}
        except Exception as e:
            last_error = e
            time.sleep(min(60, 2 ** attempt))
    raise RuntimeError(f"{request_id}: {last_error}")

requests_df = pd.read_csv("requests.csv")

done_ids = set()
if ENRICH_CACHE.exists():
    with open(ENRICH_CACHE, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                done_ids.add(json.loads(line)["request_id"])

todo = requests_df[~requests_df["request_id"].isin(done_ids)]
print(f"Already classified: {len(done_ids)}, remaining: {len(todo)}")

failures = []
with open(ENRICH_CACHE, "a", encoding="utf-8") as out, \
        concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    futures = {
        pool.submit(classify_request, row.request_id, row.description): row.request_id
        for row in todo.itertuples()
    }
    for future in tqdm.tqdm(
        concurrent.futures.as_completed(futures), total=len(futures)
    ):
        try:
            out.write(json.dumps(future.result()) + "\n")
            out.flush()
        except Exception as e:
            failures.append(str(e))

print(f"Failures: {len(failures)}")
print(failures[:5])

enriched_df = (
    pd.read_json(ENRICH_CACHE, lines=True)
    .drop_duplicates("request_id", keep="last")
)
enriched_df.to_csv("enriched.csv", index=False)
print(enriched_df.shape)
enriched_df.head()

Already classified: 0, remaining: 2000


100%|██████████| 2000/2000 [08:26<00:00,  3.95it/s]

Failures: 0
[]
(2000, 5)


,request_id,incident_type,urgency,safety_hazard,evidence
0,R000210,sanitation,urgent,yes,“A large pile of rubbish is burning next to a ...
1,R000221,drainage,urgent,yes,"Rain has flooded the bus stop, and people are ..."
2,R000103,drainage,urgent,yes,“Blocked inlet” has left the school approach “...
3,R000232,sanitation,urgent,yes,“Several used needles are scattered on the pat...
4,R000176,road,routine,yes,“The bike lane has broken pavement” causing “a...


In [10]:
validation_df = pd.read_csv("validation_labels.csv", dtype=str)
enriched_df = pd.read_csv("enriched.csv", dtype=str)
check = validation_df.merge(enriched_df, on="request_id", how="left")

print(f"Validation rows matched: {check['incident_type'].notna().sum()} / {len(check)}")
for field in ["incident_type", "urgency", "safety_hazard"]:
    accuracy = (check[f"{field}_reference"] == check[field]).mean()
    print(f"{field}: accuracy = {accuracy:.3f}")

display(pd.crosstab(check["safety_hazard_reference"], check["safety_hazard"], margins=True))
display(pd.crosstab(check["incident_type_reference"], check["incident_type"], margins=True))

check[check["safety_hazard_reference"] != check["safety_hazard"]][
    ["request_id", "safety_hazard_reference", "safety_hazard", "evidence"]
]

Validation rows matched: 200 / 200
incident_type: accuracy = 0.980
urgency: accuracy = 0.970
safety_hazard: accuracy = 0.730


safety_hazard,no,unknown,yes,All
safety_hazard_reference,,,,
no,65,4,50,119
unknown,0,20,0,20
yes,0,0,61,61
All,65,24,111,200


incident_type,drainage,electrical,other,road,sanitation,traffic_signal,tree,unknown,water,All
incident_type_reference,,,,,,,,,,
drainage,19,0,0,0,0,0,0,0,0,19
electrical,0,17,0,0,0,0,0,0,0,17
other,0,0,20,0,1,0,0,0,0,21
road,0,0,3,36,0,0,0,0,0,39
sanitation,0,0,0,0,27,0,0,0,0,27
traffic_signal,0,0,0,0,0,13,0,0,0,13
tree,0,0,0,0,0,0,21,0,0,21
unknown,0,0,0,0,0,0,0,20,0,20
water,0,0,0,0,0,0,0,0,23,23


,request_id,safety_hazard_reference,safety_hazard,evidence
3,R000703,no,yes,“Broken pavement” in the bike lane makes for “...
7,R001030,no,unknown,“The roots have started lifting the soil aroun...
10,R001318,no,yes,“The street surface is rough” beside the trans...
12,R001426,no,yes,“Water running beneath the pavement” near the ...
15,R001823,no,yes,The pedestrian countdown display has stopped s...
21,R002624,no,yes,“Hear water running beneath the pavement” when...
24,R002839,no,yes,“broken pavement” makes the bike lane “a very ...
36,R004402,no,yes,“Sidewalk slabs have become uneven” and “the s...
38,R004650,no,yes,A small hole in the service lane is “getting w...
43,R005055,no,yes,The pedestrian countdown display has stopped s...


# Part 1: Build a Data-Analysis Agent Using Tools

You will now build an agent that can analyze your enriched dataset.
This agent will decide **which data-analysis operations to perform** in response to a question.

The agent will be provided the following functions:
1. `describe_database`
2. `run_sql`
3. `create_chart`
4. `create_report`

You are given an outline of an agent, `run_analyst` that can call those functions.
Your task is to complete the integration between `run_analyst` and the functions above.


### Load Data into DuckDB

In [11]:
DB_FILE = "city_services.duckdb"

setup = duckdb.connect(DB_FILE)

setup.execute("""
    CREATE OR REPLACE TABLE requests AS
    SELECT * FROM read_csv_auto('requests.csv')
""")

setup.execute("""
    CREATE OR REPLACE TABLE districts AS
    SELECT * FROM read_csv_auto('districts.csv')
""")

setup.execute("""
    CREATE OR REPLACE TABLE events AS
    SELECT * FROM read_csv_auto('events.csv')
""")

setup.execute("""
    CREATE OR REPLACE TABLE enriched AS
    SELECT * FROM read_csv('enriched.csv', header = true, all_varchar = true)
""")

setup.execute("""
    CREATE OR REPLACE TABLE validation_labels AS
    SELECT * FROM read_csv('validation_labels.csv', header = true, all_varchar = true)
""")

setup.execute("""
    CREATE OR REPLACE TABLE requests_enriched AS
    SELECT
        r.*,
        e.incident_type AS llm_incident_type,
        e.urgency AS llm_urgency,
        e.safety_hazard AS llm_safety_hazard,
        e.evidence AS llm_evidence,
        d.district_name,
        d.population,
        date_diff('minute', r.created_at, r.closed_at) / 60.0 AS hours_to_close
    FROM requests r
    LEFT JOIN enriched e USING (request_id)
    LEFT JOIN districts d USING (district_id)
""")

print(setup.execute("SELECT COUNT(*), COUNT(llm_safety_hazard) FROM requests_enriched").fetchall())

setup.close()

[(2000, 2000)]


### Setup Read-Only Data Connection

In [12]:
con = duckdb.connect(
    DB_FILE,
    read_only=True,
    config={
        "enable_external_access": "false",
        "autoload_known_extensions": "false",
        "autoinstall_known_extensions": "false",
        "allow_community_extensions": "false",
        "allow_unsigned_extensions": "false",
        "lock_configuration": "true",
        "memory_limit": "512MB",
        "threads": "2",
    },
)

### Tool: describe_database

In [13]:
def describe_database(table: str | None = None) -> dict:
    """List DuckDB tables or describe a specific table."""
    try:
        # Discover available tables.
        tables = con.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
              AND table_type IN ('BASE TABLE', 'VIEW')
            ORDER BY table_name
        """).fetchall()

        table_names = [row[0] for row in tables]

        if table is not None and table not in table_names:
            return {
                "ok": False,
                "error": f"Unknown table: {table}",
                "available_tables": table_names,
            }

        def row_count(name):
            # Names are checked against the database catalog.
            quoted = '"' + name.replace('"', '""') + '"'
            return con.execute(
                f"SELECT COUNT(*) FROM {quoted}"
            ).fetchone()[0]

        # No table specified: list all tables.
        if table is None:
            return {
                "ok": True,
                "tables": [
                    {
                        "name": name,
                        "row_count": row_count(name),
                    }
                    for name in table_names
                ],
            }

        # Describe the specified table.
        columns = con.execute("""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'main'
              AND table_name = ?
            ORDER BY ordinal_position
        """, [table]).fetchall()

        return {
            "ok": True,
            "table": table,
            "row_count": row_count(table),
            "columns": [
                {"name": name, "type": dtype}
                for name, dtype in columns
            ],
        }

    except Exception as e:
        return {"ok": False, "error": str(e)}


### Tool: run_sql

In [14]:
MAX_SQL_ROWS = 100

def _execute_select(query, max_rows):
    """
    Execute one SELECT statement against DuckDB.

    Returns (columns, rows, truncated).

    Raises an exception for invalid SQL or
    disallowed statement types.
    """
    if not isinstance(query, str) or not query.strip():
        raise ValueError("SQL query must be a nonempty string.")

    statements = con.extract_statements(query)

    if (len(statements) != 1 or
        statements[0].type != duckdb.StatementType.SELECT):
        raise ValueError("Only one SELECT statement is allowed.")

    result = con.execute(query)
    columns = [col[0] for col in result.description]

    if len(set(columns)) != len(columns):
        raise ValueError(
            "Duplicate column names. Use SQL aliases."
        )

    # Fetch one extra row to detect truncation.
    rows = result.fetchmany(max_rows + 1)
    truncated = len(rows) > max_rows

    return columns, rows[:max_rows], truncated

def _json_value(value):
    """Convert common DuckDB values to JSON-safe values."""
    if isinstance(value, (datetime.date, datetime.datetime, datetime.time)):
        return value.isoformat()
    if isinstance(value, decimal.Decimal):
        return float(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, (list, tuple)):
        return [_json_value(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _json_value(v) for k, v in value.items()}
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)

def run_sql(query: str) -> dict:
    """
    Execute a read-only DuckDB SELECT query.

    Returns up to 100 rows as JSON-compatible data.
    """
    try:
        columns, rows, truncated = _execute_select(
            query, MAX_SQL_ROWS
        )

        return {
            "ok": True,
            "columns": columns,
            "rows": [
                {
                    col: _json_value(value)
                    for col, value in zip(columns, row)
                }
                for row in rows
            ],
            "row_count": len(rows),
            "truncated": truncated,
        }

    except Exception as e:
        return {
            "ok": False,
            "error": str(e),
        }

### Tool: create_chart

In [15]:
CHART_DIR = pathlib.Path("charts")
CHART_DIR.mkdir(exist_ok=True)

MAX_CHART_POINTS = 500

def create_chart(
    query: str,
    filename: str,
    chart_type: str = "plot",
    fmt: str = "-o",
    title: str = "",
    xlabel: str = "",
    ylabel: str = "",
    rotation: int = 0,
) -> dict:
    """
    Execute SQL returning columns x and y, then plot.

    chart_type:
        "plot" -> matplotlib.pyplot.plot(x, y, fmt)
        "bar"  -> matplotlib.pyplot.bar(x, y)

    Saves the result as a PNG in the charts directory.
    """
    fig = None

    try:
        if chart_type not in ("plot", "bar"):
            raise ValueError(
                "chart_type must be 'plot' or 'bar'."
            )

        if not isinstance(filename, str) or not re.fullmatch(
            r"[A-Za-z0-9][A-Za-z0-9_.-]*\.png", filename
        ):
            raise ValueError(
                "filename must be a simple PNG filename."
            )

        if not -90 <= rotation <= 90:
            raise ValueError(
                "rotation must be between -90 and 90."
            )

        columns, rows, truncated = _execute_select(
            query, MAX_CHART_POINTS
        )

        if columns != ["x", "y"]:
            raise ValueError(
                "Query must return exactly two columns "
                "named x and y."
            )

        if truncated:
            raise ValueError(
                "Too many points. Aggregate or filter "
                "the query to at most 500 rows."
            )

        if not rows:
            raise ValueError("Query returned no data.")

        x = [row[0] for row in rows]
        y = [row[1] for row in rows]

        if any(v is None for v in x):
            raise ValueError("x contains NULL values.")

        if any(
            isinstance(v, bool)
            or not isinstance(v, (int, float, decimal.Decimal))
            or not math.isfinite(v)
            for v in y
        ):
            raise ValueError(
                "y must contain finite numeric values."
            )

        # Thin wrapper around Matplotlib.
        fig = plt.figure(figsize=(8, 5))

        if chart_type == "plot":
            plt.plot(x, y, fmt)
        else:
            plt.bar(x, y)

        plt.title(title)
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.xticks(rotation=rotation)
        plt.tight_layout()

        path = CHART_DIR / filename
        plt.savefig(path)

        return {
            "ok": True,
            "filename": str(path),
            "chart_type": chart_type,
            "points_plotted": len(rows),
        }

    except Exception as e:
        return {
            "ok": False,
            "error": str(e),
        }

    finally:
        if fig is not None:
            plt.close(fig)

### Tool: create_report

In [16]:
def create_report(filename: str, text: str) -> dict:
    """Write Markdown text to a new or overwritten .md file."""
    try:
        if "/" in filename or "\\" in filename:
            raise ValueError("Filename must not contain slashes.")

        if not filename.endswith(".md"):
            raise ValueError("Filename must end in '.md'.")

        with open(filename, "w", encoding="utf-8") as f:
            f.write(text)

        return {
            "ok": True,
            "filename": filename
        }

    except (ValueError, OSError) as e:
        return {
            "ok": False,
            "error": str(e)
        }

### Analyst Agent

The `tools` list provides tool definitions to pass to the API.
The definition for the `describe_database` tool has been provided.
Fill in the tool definitions for the remaining tools defined above.
Note that your descriptions here will influence how and when the agent decides to use these tools.

In [17]:
# YOUR CHANGES HERE

tools = [
    {
        "type": "function",
        "name": "describe_database",
        "description": (
            "List the available DuckDB tables or describe the "
            "columns and types of a specified table. "
            "Use this tool to discover the database structure "
            "before writing SQL queries."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "table": {
                    "type": ["string", "null"],
                    "description": (
                        "Name of the table to describe, "
                        "or null to list all tables."
                    )
                }
            },
            "required": ["table"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "run_sql",
        "description": (
            "Run exactly one read-only DuckDB SELECT statement and return the result "
            "as JSON rows (at most 100 rows; 'truncated' is true if more existed). "
            "Use this for every number you report: counts, rates, averages, percentages, "
            "cross-tabulations. Aggregate in SQL (GROUP BY, COUNT, AVG, ROUND) instead of "
            "fetching raw rows. Give every output column a unique alias. "
            "Only SELECT is allowed; CREATE, INSERT, UPDATE, COPY and multiple statements fail."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "A single DuckDB SELECT statement."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "create_chart",
        "description": (
            "Run a DuckDB SELECT query that returns exactly two columns aliased x and y, "
            "then draw a matplotlib chart and save it as charts/<filename>. "
            "x is the category or time value; y must be numeric and non-null. "
            "Use chart_type 'bar' for comparisons across categories such as districts, "
            "and 'plot' for trends over time. At most 500 rows; aggregate first. "
            "Before charting, verify the same numbers with run_sql. "
            "Embed the saved chart in the report as ![caption](charts/<filename>)."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "SELECT statement returning only columns named x and y."
                },
                "filename": {
                    "type": "string",
                    "description": "Simple PNG file name such as hazard_rate_by_district.png (no folders)."
                },
                "chart_type": {
                    "type": "string",
                    "enum": ["plot", "bar"],
                    "description": "'bar' for categories, 'plot' for a line over time."
                },
                "fmt": {
                    "type": "string",
                    "description": "Matplotlib format string for 'plot' charts, e.g. '-o'. Use '-o' for bar charts."
                },
                "title": {
                    "type": "string",
                    "description": "Chart title stating what is measured."
                },
                "xlabel": {
                    "type": "string",
                    "description": "X-axis label."
                },
                "ylabel": {
                    "type": "string",
                    "description": "Y-axis label including units, e.g. 'Hazard requests per 10,000 residents'."
                },
                "rotation": {
                    "type": "integer",
                    "description": "X tick label rotation in degrees between -90 and 90; use 30-45 for long names."
                }
            },
            "required": ["query", "filename", "chart_type", "fmt", "title", "xlabel", "ylabel", "rotation"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "create_report",
        "description": (
            "Write the final Markdown report to a .md file in the working directory, "
            "overwriting any existing file of that name. Call this once all numbers have "
            "been computed with run_sql and all charts created with create_chart. "
            "The text must be complete Markdown; reference charts as ![caption](charts/<filename>)."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "File name ending in .md with no slashes, e.g. report.md."
                },
                "text": {
                    "type": "string",
                    "description": "The full Markdown content of the report."
                }
            },
            "required": ["filename", "text"],
            "additionalProperties": False
        },
        "strict": True
    },
]

The functions dictionary is used to map responses requesting function invocations back to Python functions.

In [18]:
functions = {
    "describe_database": describe_database,
    "run_sql": run_sql,
    "create_chart": create_chart,
    "create_report": create_report,
}

In [19]:
def log_event(file, event, **data):
    """Append one JSON event to an open log file."""
    record = {
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "event": event,
        **data,
    }
    file.write(json.dumps(record, default=str) + "\n")
    file.flush()

The `run_analyst` function below provides an agent loop calling the Responses API, logging the results, and invoking tools.
Most of the function is complete, but it lacks instructions to the agent to steer its behavior.
You will need to fill in the missing instructions.

In [20]:
ANALYST_INSTRUCTIONS = """
You are a careful data analyst for a city's public works department. You answer questions
by querying a read-only DuckDB database through tools.

Database notes:
- requests: one row per service request. incident_id groups duplicate reports of the same
  incident; closure_reason = 'Duplicate' marks duplicates. intake_category and intake_priority
  are the city's original intake labels and are known to be imperfect.
- enriched: LLM-derived labels per request_id (incident_type, urgency, safety_hazard, evidence).
  Values are text: safety_hazard is 'yes', 'no' or 'unknown'.
- requests_enriched: requests joined with enriched labels (prefixed llm_), district_name,
  population, and hours_to_close (NULL when the request is still open).
- districts: district names, population, area, road miles, income, housing units.
- events: weather and operational events by district and date range, including policy changes
  such as a mobile reporting rollout, a staffing expansion and a triage workflow update.
- validation_labels: hand-labelled reference values for a sample of requests, usable to
  measure how accurate the LLM labels are.

Working rules:
1. Start with describe_database to confirm tables and columns before writing SQL.
2. Every number you state must come from a tool result in this conversation. Never estimate,
   round differently from the tool output, or invent values. If a query fails, fix it and rerun.
3. Do the arithmetic in SQL (counts, percentages, per-10,000-resident rates, averages, medians).
   Show denominators alongside rates.
4. Compare districts with population-normalized rates, not just raw counts.
5. Consider duplicates: report whether counting distinct incident_id changes a conclusion.
6. Treat 'unknown' LLM labels explicitly; do not silently fold them into 'yes' or 'no'.
7. When looking at events, align dates and district_id, and remember that a change in reporting
   channels or workflow can change counts without any change in real conditions.
8. Describe associations as associations. Do not claim causation.
9. For charts, verify the numbers with run_sql first, then call create_chart with columns
   aliased x and y, readable titles and labelled axes.
10. When asked for a report, write it with create_report, embed every chart as
    ![caption](charts/<filename>), and then reply with a brief summary of what the file contains.
11. Keep answers concise and quantitative.
"""

In [21]:
# YOUR CHANGES HERE

def run_analyst(
    prompt,
    model="gpt-6-luna",
    max_steps=30,
    log_filename="analyst_log.jsonl",
):
    run_id = str(uuid.uuid4())
    previous_response_id = None
    next_input = prompt

    total_tokens = 0
    total_tool_calls = 0

    with open(log_filename, "a", encoding="utf-8") as log:

        def record(event, **data):
            log_event(log, event, run_id=run_id, **data)

        record("run_start", prompt=prompt, model=model)

        for step in tqdm.tqdm(range(1, max_steps + 1)):
            # Call the model.
            start = time.perf_counter()

            try:
                kwargs = {}
                if previous_response_id is not None:
                    kwargs["previous_response_id"] = previous_response_id

                response = client.responses.create(
                    model=model,
                    input=next_input,
                    tools=tools,
                    instructions=ANALYST_INSTRUCTIONS,
                    **kwargs,
                )
            except Exception as e:
                record("api_error", step=step, error=str(e))
                raise

            elapsed = time.perf_counter() - start

            usage = response.usage
            if usage is not None:
                total_tokens += usage.total_tokens

            record(
                "model_response",
                step=step,
                response_id=response.id,
                status=response.status,
                elapsed_seconds=elapsed,
                input_tokens=usage.input_tokens if usage else None,
                output_tokens=usage.output_tokens if usage else None,
            )

            if response.status != "completed":
                record("run_end", status=response.status)
                raise RuntimeError(
                    f"Model response status: {response.status}"
                )

            calls = [
                item for item in response.output
                if item.type == "function_call"
            ]

            # The agent has finished.
            if not calls:
                answer = response.output_text

                record(
                    "run_end",
                    status="completed",
                    answer=answer,
                    total_tokens=total_tokens,
                    total_tool_calls=total_tool_calls,
                )

                return IPython.display.display(IPython.display.Markdown(answer))

            # Execute all requested tool calls.
            outputs = []

            for call in calls:
                total_tool_calls += 1

                record(
                    "tool_call",
                    step=step,
                    call_id=call.call_id,
                    name=call.name,
                    arguments=call.arguments,
                )

                start = time.perf_counter()

                try:
                    args = json.loads(call.arguments)
                    result = functions[call.name](**args)
                except Exception as e:
                    result = {"ok": False, "error": str(e)}

                elapsed = time.perf_counter() - start

                record(
                    "tool_result",
                    step=step,
                    call_id=call.call_id,
                    name=call.name,
                    result=result,
                    elapsed_seconds=elapsed,
                )

                outputs.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(result, default=str),
                })

            # Continue the conversation using the tool results.
            previous_response_id = response.id
            next_input = outputs

        record(
            "run_end",
            status="max_steps_exceeded",
            total_tokens=total_tokens,
            total_tool_calls=total_tool_calls,
        )

        raise RuntimeError("Agent exceeded maximum steps.")


In [22]:
run_analyst("how many districts are there?")

  3%|▎         | 1/30 [00:01<00:30,  1.07s/it]

There are **8 districts**.

  3%|▎         | 1/30 [00:02<01:14,  2.58s/it]


# Part 2: Produce an Infrastructure Safety Report

The city manager is concerned that potentially dangerous infrastructure incidents are not always being identified or handled consistently. You have been asked to prepare an evidence-based report.

Use `run_analyst` to analyze the data given to identify patterns in potentially hazardous incidents.
Identify any significant concentrations, and back them up with quantitative evidence.
Generate a report `report.md` with at least 3 charts supporting your conclusions.

Your final report must include:

1. **Executive summary:** The main findings in a short paragraph.
2. **Quantitative results:** At least three specific findings, with values that can be reproduced from the data.
3. **Visualizations:** At least two charts produced using your agent's tools.
4. **Classification analysis:** At least one comparison between the original intake classifications and your LLM-derived fields.
5. **Caveats:** Explain how uncertain labels, missing data, differences in district populations, and incomplete service requests might affect the conclusions.
6. **Recommendation:** Suggest one concrete issue the city should investigate further, without presenting correlation as proof of causation.

You may interact with your agent, ask follow-up questions, and request revisions. However, the reported numbers must come from tool outputs rather than being invented by the language model.

**Make sure to check your report before submission; it will be manually graded by the teaching staff.**

In [23]:
# YOUR CHANGES HERE

long_prompt = """
The city manager is concerned that potentially dangerous infrastructure incidents are not always
identified or handled consistently. Investigate the data and write an evidence-based report to
report.md using create_report.

Investigate at least the following, using SQL for every number:
A. Overall: how many requests the LLM labels as safety_hazard = 'yes', 'no', 'unknown', and how
   many distinct incidents these represent once duplicates are accounted for.
B. Geography: hazard requests by district as counts and as rates per 10,000 residents, and the
   share of each district's requests that are hazards. Identify any district that stands out.
C. Intake vs LLM: cross-tabulate intake_priority against llm_safety_hazard and llm_urgency.
   Report how many and what share of LLM-flagged hazards were given intake_priority 'Routine' or
   'Unassigned'. Also compare intake_category with llm_incident_type and find the categories with
   the most disagreement.
D. Handling: compare hours_to_close for hazard vs non-hazard requests (average and median),
   overall and by district, and by intake_priority within hazards.
E. Time and events: monthly counts of hazard requests overall and for any standout district.
   Check the events table for events in that district (rainstorms, mobile reporting rollout,
   staffing expansion, triage workflow update) and compare before vs after periods using dates.
   Also check report_channel to see whether a new channel explains a change in volume.
F. Label quality: using validation_labels joined to enriched on request_id, compute the accuracy
   of the LLM labels for incident_type, urgency and safety_hazard, plus a confusion table for
   safety_hazard.

Create at least 3 charts with create_chart, for example:
- hazard requests per 10,000 residents by district (bar)
- monthly hazard requests for the standout district or the whole city (plot)
- share of LLM-flagged hazards by intake_priority, or median hours to close for hazards by district (bar)

The report must contain these sections in this order:
1. Executive Summary: one short paragraph with the main findings.
2. Quantitative Results: at least three specific findings with exact values, denominators, and a
   one-line description of how each was computed so it can be reproduced.
3. Visualizations: the charts embedded as ![caption](charts/<filename>) with a sentence each
   explaining what it shows.
4. Classification Analysis: the intake vs LLM comparison, plus the validation accuracy numbers.
5. Caveats: how uncertain or 'unknown' labels, LLM labelling error, missing data (open requests
   with no closed_at), different district populations, duplicate requests, changes in reporting
   channels, and incomplete descriptions could affect the conclusions.
6. Recommendation: one concrete issue the city should investigate further, framed as something to
   investigate, not as a proven cause.

Use only numbers returned by your tools. After writing report.md, reply with a short summary.
"""
run_analyst(long_prompt, max_steps=60, log_filename="report_log.jsonl")

 15%|█▌        | 9/60 [01:14<12:32, 14.75s/it]

Created **report.md** with the requested quantitative findings, classification and validation analysis, caveats, recommendation, and three embedded charts. It highlights Eastbank’s elevated population-normalized hazard request rate and the association between the May reporting rollout and increased reports, while noting that neither confirms a change in underlying conditions.

 15%|█▌        | 9/60 [01:17<07:19,  8.61s/it]


In [24]:
IPython.display.display(IPython.display.Markdown(pathlib.Path("report.md").read_text(encoding="utf-8")))
print(sorted(p.name for p in CHART_DIR.glob("*.png")))

# Potentially Dangerous Infrastructure Incidents: Evidence Review

## 1. Executive Summary

The LLM flags 1,137 of 2,000 requests as hazards, representing 1,131 distinct incidents; validation accuracy for the safety label is 73.0%, so these flags warrant verification rather than being treated as confirmed hazards. Eastbank has the highest population-normalized hazard volume (26.33 requests per 10,000 residents), while 822/1,137 flagged requests carry an intake priority of Routine or Unassigned. Hazard-flagged requests also take longer to close than the LLM-labeled “no” group. Eastbank’s reported volume rises after a mobile reporting rollout, but the timing coincides with channel changes and rainstorms and does not establish a cause.

## 2. Quantitative Results

1. **Overall labels and duplicates.** Of 2,000 requests, 1,137 were labeled safety hazard “yes” (56.85%), 734 “no,” and 129 “unknown.” These represent, respectively, 1,131, 733, and 128 distinct incident IDs. Thus, counting distinct incidents rather than requests reduces the “yes” total by 6; duplicate reports are present and requests should not be read as unique incidents. *Reproduction: grouped requests_enriched by llm_safety_hazard and counted rows and distinct incident_id.*

2. **Geographic concentration.** Eastbank has 237 hazard requests among 388 requests and population 90,000: 26.33 requests per 10,000 residents and a 61.08% within-district request share. It remains the highest district if deduplicated: 233 distinct hazard incidents, or 25.89 per 10,000 residents. Old Port has the largest share of its requests labeled hazard (187/250 = 74.80%), but a lower normalized count rate (22.26 requests per 10,000 residents). *Reproduction: counted hazard requests and divided by district population for the rate; divided hazard requests by all district requests for the share; also counted distinct incident_id.*

   | District | Hazard requests / all requests | Population | Hazard requests per 10,000 residents | Hazard share of requests |
   |---|---:|---:|---:|---:|
   | Eastbank | 237 / 388 | 90,000 | 26.33 | 61.08% |
   | Old Port | 187 / 250 | 84,000 | 22.26 | 74.80% |
   | West Commons | 122 / 200 | 68,000 | 17.94 | 61.00% |
   | Southpoint | 161 / 308 | 95,000 | 16.95 | 52.27% |
   | Riverside | 120 / 227 | 79,000 | 15.19 | 52.86% |
   | North Harbor | 105 / 198 | 74,000 | 14.19 | 53.03% |
   | Cedar Hills | 86 / 187 | 61,000 | 14.10 | 45.99% |
   | Lakeview | 119 / 242 | 85,000 | 14.00 | 49.17% |

3. **Intake priority versus hazard flag.** Among the 1,137 LLM-flagged hazard requests, 822 (72.30%) have intake priority Routine or Unassigned: 759 Routine and 63 Unassigned. The other 315 are High. The cross-tab also shows 530 hazard requests labeled “urgent” by the LLM and 607 labeled “routine” (0 with unknown urgency). *Reproduction: filtered to safety_hazard = yes, counted by intake_priority and llm_urgency, and divided Routine plus Unassigned by all hazard requests.*

4. **Handling time.** Among closed requests, hazard “yes” has mean time to close 62.11 hours and median 23.75 hours (1,132 closed of 1,137); hazard “no” has mean 26.05 and median 13.82 hours (734/734 closed). The separate “unknown” group has mean 45.35 and median 17.50 hours (128/129 closed). *Reproduction: averaged and took the median of non-null hours_to_close within each label; open requests are excluded from these duration statistics.*

   | District | Hazard yes: n closed, mean / median hours | Hazard no: n closed, mean / median hours |
   |---|---:|---:|
   | Cedar Hills | 86, 97.47 / 22.97 | 95, 28.27 / 16.72 |
   | Eastbank | 237, 49.00 / 17.10 | 130, 21.08 / 10.72 |
   | Lakeview | 118, 46.64 / 25.33 | 105, 23.42 / 16.20 |
   | North Harbor | 104, 69.27 / 21.97 | 82, 22.55 / 13.57 |
   | Old Port | 186, 96.73 / 74.65 | 50, 65.52 / 17.32 |
   | Riverside | 119, 35.50 / 20.18 | 94, 23.37 / 13.64 |
   | Southpoint | 160, 42.53 / 18.98 | 116, 22.85 / 12.64 |
   | West Commons | 122, 70.39 / 23.01 | 62, 20.41 / 12.56 |

   Within hazard “yes” requests, High priority has 315 closed requests, mean 92.81 and median 54.60 hours; Routine has 755 closed and 4 open, mean 49.88 and median 19.87 hours; Unassigned has 62 closed and 1 open, mean 55.12 and median 36.20 hours. *Reproduction: grouped hazard requests by intake_priority and calculated mean and median over non-null hours_to_close.*

5. **Time and events in the standout district.** Eastbank recorded 42 hazard requests among 79 total requests in January–April, before the May 1 mobile reporting rollout (10.50 hazard requests per month); from May–December it recorded 195 among 309 requests (24.38 per month). The post-rollout period includes 191 distinct hazard incidents. The hazard share was 53.16% before and 63.11% in the rollout period. Monthly hazard request counts were 10, 12, 5, 15, 33, 30, 27, 28, 19, 21, 19, and 18 from January through December, respectively. *Reproduction: joined events to the district by district_id and compared request created_at before May 1, 2025, with dates from May 1 onward; counted monthly hazard requests and divided period hazard counts by the number of months.*

   Eastbank’s event records include rainstorms on March 23–26, May 12–16 (intensity 5), September 5–8, and September 20–21. The mobile reporting rollout ran May 1–December 31. No staffing expansion or triage workflow update is listed for Eastbank; those events are listed for Riverside (staffing expansion from June 16) and Southpoint (triage update from September 1). Channel mix changes markedly across the Eastbank periods: mobile reports were 3/79 (3.80%) before and 230/309 (74.43%) in the rollout period; phone counts were 39 and 29, web counts 37 and 50. This association makes a reporting-volume change plausible to investigate, but the concurrent rainstorm and reporting changes cannot identify what produced the counts.

## 3. Visualizations

![LLM-flagged hazard requests per 10,000 residents by district](charts/hazard_rate_district.png)

Eastbank has the highest population-normalized hazard request count; the rate is based on requests, with population as denominator.

![Monthly LLM-flagged hazard requests in Eastbank, 2025](charts/eastbank_monthly_hazards.png)

Eastbank’s monthly series rises in May, the month the mobile rollout began, and remains above the first-quarter monthly counts afterward; counts are not proof of a change in underlying conditions.

![Median time to close hazard-flagged requests by district](charts/median_close_hazard_by_district.png)

Median handling duration varies by district; Old Port is notably higher than the other district medians.

## 4. Classification Analysis

**Intake versus LLM labels.** Intake priority does not consistently surface LLM hazard flags: 759 Routine and 63 Unassigned requests are hazard-flagged (822/1,137, 72.30%). Within the cross-tab, Routine priority includes 517 hazard requests labeled LLM urgency “routine” and 242 labeled “urgent”; Unassigned includes 22 and 41, respectively. High priority includes 68 hazard requests labeled “routine” and 247 “urgent.” These are request counts, not confirmed dangerous incidents.

For intake category versus LLM incident type, literal text comparisons show the largest mismatch counts for Road Maintenance (632/632), General Inquiry (274/274), Streetlight (175/175), and Water Service (145/145). These are **literal mismatches**, not necessarily semantic disagreements: for example, the LLM uses labels such as “road” and “electrical,” which are not exact matches for the intake labels. Drainage has 58 mismatches among 177 requests (32.77%), and Sanitation has 34 among 286 (11.89%). A reviewed crosswalk is needed before interpreting these as classification errors.

**Validation against hand labels.** In the joined validation sample (n = 200), exact-match accuracy is 196/200 (98.0%) for incident type, 194/200 (97.0%) for urgency, and 146/200 (73.0%) for safety hazard. The observed safety-hazard confusion pairs (LLM label → reference label) are: no → no, 65; unknown → no, 4; unknown → unknown, 20; yes → no, 50; and yes → yes, 61. Thus, among the validation examples, many predicted “yes” labels are not “yes” in the reference, while unknown labels also occur. *Reproduction: joined validation_labels to enriched on request_id, counted exact label matches for accuracy, and grouped the safety labels by predicted and reference values.*

## 5. Caveats

- The “unknown” label is material: 129 requests are unknown for safety, and validation includes 20 unknown/unknown pairs and 4 unknown/no pairs. Unknown should remain a separate review category, not be assumed safe or hazardous.
- Safety-label accuracy is 73.0% in the 200-request hand-labeled sample. The sample may not represent every request, and both false-positive and missed-hazard risk remain; LLM flags are not confirmed findings.
- Six requests have no closed_at (2,000 total requests); five are hazard “yes” and one is “unknown.” Open cases have no hours_to_close and are omitted from duration averages and medians, so unresolved cases may change the handling picture.
- District populations differ. The district rate table therefore uses population denominators; request shares answer a different question and should not be used alone to compare hazard burden. Rates are based on population rather than road miles or infrastructure exposure.
- Duplicate reports affect request totals: hazard “yes” is 1,137 requests but 1,131 distinct incident IDs. District and time comparisons can also be affected by duplicates; Eastbank’s monthly incident counts differ from request counts in July and November.
- Reporting channels changed in Eastbank at the same time as the mobile rollout. More submitted requests may reflect access/reporting changes, not necessarily more underlying incidents. Dates also overlap with rainstorms; these are associations, not causal evidence.
- Intake categories and LLM incident-type labels use different vocabularies. Exact-string mismatch rates overstate semantic disagreement unless a category crosswalk is applied. Incomplete or ambiguous descriptions may also limit LLM classification; this analysis does not independently assess description completeness or confirm infrastructure conditions.

## 6. Recommendation

Investigate whether Eastbank’s increased post-May hazard-report volume and the 822 Routine/Unassigned hazard flags reflect missed intake triage, the new mobile reporting channel, storm-related conditions, or a combination. Review a stratified sample of Eastbank reports and their descriptions, intake decisions, field outcomes, and closure records against hand-labeled safety criteria; prioritize the safety-label validation gaps before using LLM flags as operational determinations. This is a follow-up question, not a conclusion about cause.

['eastbank_monthly_hazards.png', 'hazard_rate_district.png', 'median_close_hazard_by_district.png']


# Part 3: Code

Submit your Jupyter notebook, `report.md` produced by `run_analyst`, all of its embedded images, and the report-generation transcript `report_log.jsonl` to Gradescope.

# Part 4: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.